<a href="https://colab.research.google.com/github/johnkarigi/AVIATION-ACCIDENT-ANALYSIS/blob/main/p2_code_challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 Code Challenge

This code challenge is designed to test your understanding of the Phase 2 material. It covers:

- SQL
- Bayesian Statistics
- Normal Distribution
- Statistical Tests

_Read the instructions carefully_. You will be asked both to write code and to answer short answer questions.

## Code Tests

We have provided some code tests for you to run to check that your work meets the item specifications. Passing these tests does not necessarily mean that you have gotten the item correct - there are additional hidden tests. However, if any of the tests do not pass, this tells you that your code is incorrect and needs changes to meet the specification. To determine what the issue is, read the comments in the code test cells, the error message you receive, and the item instructions.

## Short Answer Questions

For the short answer questions...

* _Use your own words_. It is OK to refer to outside resources when crafting your response, but _do not copy text from another source_.

* _Communicate clearly_. We are not grading your writing skills, but you can only receive full credit if your teacher is able to fully understand your response.

* _Be concise_. You should be able to answer most short answer questions in a sentence or two. Writing unnecessarily long answers increases the risk of you being unclear or saying something incorrect.

In [ ]:
# Run this cell without changes to import the necessary libraries

import itertools
import numpy as np
import pandas as pd
from numbers import Number
import sqlite3
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import pickle

---
## Part 1: SQL [Suggested time: 20 minutes]
---
In this part, you will create and execute three SQL queries on the Chinook database. For this challenge **you will need to access the `Album` and `Artist` tables**.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Chinook_Sqlite.sqlite to Chinook_Sqlite.sqlite


### 1.1) Connect to the Database.

In [ ]:
# CodeGrade step1.1
# Replace None with appropriate code
# Connect to the Database here ("Chinook_Sqlite.sqlite")

path = "Chinook_Sqlite.sqlite"
conn = sqlite3.connect(path)

In [ ]:
assert type(path) == str

In [ ]:
# Run this cell without changes to see all the
# tables in the database.

df = pd.read_sql(
    """
    SELECT *
    FROM sqlite_master
    """
, conn
)

df[df['type'] == 'table']

,type,name,tbl_name,rootpage,sql
0,table,Album,Album,2,CREATE TABLE [Album]\n(\n [AlbumId] INTEGER...
1,table,Artist,Artist,3,CREATE TABLE [Artist]\n(\n [ArtistId] INTEG...
2,table,Customer,Customer,4,CREATE TABLE [Customer]\n(\n [CustomerId] I...
3,table,Employee,Employee,7,CREATE TABLE [Employee]\n(\n [EmployeeId] I...
4,table,Genre,Genre,9,CREATE TABLE [Genre]\n(\n [GenreId] INTEGER...
5,table,Invoice,Invoice,10,CREATE TABLE [Invoice]\n(\n [InvoiceId] INT...
6,table,InvoiceLine,InvoiceLine,12,CREATE TABLE [InvoiceLine]\n(\n [InvoiceLin...
7,table,MediaType,MediaType,14,CREATE TABLE [MediaType]\n(\n [MediaTypeId]...
8,table,Playlist,Playlist,15,CREATE TABLE [Playlist]\n(\n [PlaylistId] I...
9,table,PlaylistTrack,PlaylistTrack,16,CREATE TABLE [PlaylistTrack]\n(\n [Playlist...


### 1.2) Write a query to return the last ten artists alphabetically.

In [ ]:
# CodeGrade step1.2
# Replace None with appropriate code
# Hint: Use the Artist table!

first_query ="""
SELECT *
FROM Artist
ORDER BY Name DESC
LIMIT 10
"""
pd.read_sql(first_query, conn)

,ArtistId,Name
0,155,Zeca Pagodinho
1,168,Youssou N'Dour
2,212,Yo-Yo Ma
3,255,Yehudi Menuhin
4,181,Xis
5,211,Wilhelm Kempff
6,154,Whitesnake
7,73,Vinícius E Qurteto Em Cy
8,74,Vinícius E Odette Lara
9,71,Vinícius De Moraes & Baden Powell


In [ ]:
# first_query should be a string
assert type(first_query) == str

# first_query should be a SQL query
first_query_df = pd.read_sql(first_query, conn)

### 1.3) Write a query to return all the albums in the database from Led Zeppelin.

In [ ]:
# CodeGrade step1.3
# Replace None with appropriate code
# Hint: Use the Artist and Album tables!

second_query = """
SELECT Album.AlbumId, Album.Title, Artist.Name AS ArtistName
FROM Album
JOIN Artist
ON Album.ArtistId = Artist.ArtistId
WHERE Artist.Name = 'Led Zeppelin'
"""
pd.read_sql(second_query, conn)

,AlbumId,Title,ArtistName
0,30,BBC Sessions [Disc 1] [Live],Led Zeppelin
1,44,Physical Graffiti [Disc 1],Led Zeppelin
2,127,BBC Sessions [Disc 2] [Live],Led Zeppelin
3,128,Coda,Led Zeppelin
4,129,Houses Of The Holy,Led Zeppelin
5,130,In Through The Out Door,Led Zeppelin
6,131,IV,Led Zeppelin
7,132,Led Zeppelin I,Led Zeppelin
8,133,Led Zeppelin II,Led Zeppelin
9,134,Led Zeppelin III,Led Zeppelin


In [ ]:
# second_query should be a string
assert type(second_query) == str

# second_query should be a SQL query
second_query_df = pd.read_sql(second_query, conn)

### 1.4) Write a query to return both the artist with the most albums in the database and the number of albums.

In [ ]:
# CodeGrade step1.4
# Replace None with appropriate code

third_query = """
SELECT Artist.Name AS ArtistName, COUNT(Album.AlbumId) AS AlbumCount
FROM Album
JOIN Artist
ON Album.ArtistId = Artist.ArtistId
GROUP BY Artist.ArtistId, Artist.Name
ORDER BY AlbumCount DESC
LIMIT 1
"""

pd.read_sql(third_query, conn)

,ArtistName,AlbumCount
0,Iron Maiden,21


In [ ]:
# third_query should be a string
assert type(third_query) == str

# third_query should be a SQL query
third_query_df = pd.read_sql(third_query, conn)

---
## Part 2: Bayesian Statistics [Suggested time: 15 minutes]
---

A medical test is designed to diagnose a certain disease. The test has a false positive rate of 10%, meaning that 10% of people without the disease will get a positive test result. The test has a false negative rate of 2%, meaning that 2% of people with the disease will get a negative result. Only 1% of the population has this disease.

### 2.1) Create a numeric variable `p_pos_test` containing the probability of a person receiving a positive test result.

Assume that the person being tested is randomly selected from the broader population.

In [ ]:
# CodeGrade step2.1
# Replace None with appropriate code

false_pos_rate = 0.1
false_neg_rate = 0.02
population_rate = 0.01

p_pos_test = (1 - false_neg_rate) * population_rate + false_pos_rate * (1 - population_rate)
print(p_pos_test)

0.10880000000000001


In [ ]:
# This test confirms that you have created a numeric variable named p_pos_test

assert isinstance(p_pos_test, Number)

In [ ]:
# These tests confirm that p_pos_test is a value between 0 and 1

assert p_pos_test >= 0
assert p_pos_test <= 1

### 2.2) Create a numeric variable `p_disease_given_pos` containing the probability of a person actually having the disease if they receive a positive test result.

Assume that the person being tested is randomly selected from the broader population.

Hint: Use your answer to the previous question to help answer this one.

In [ ]:
# CodeGrade step2.2
# Replace None with appropriate code

false_pos_rate = 0.1
false_neg_rate = 0.02
population_rate = 0.01

p_disease_given_pos =  ((1 - false_neg_rate) * population_rate) / p_pos_test

In [ ]:
# This test confirms that you have created a numeric variable named p_disease_given_pos

assert isinstance(p_disease_given_pos, Number)

In [ ]:
# These tests confirm that p_disease_given_pos is a value between 0 and 1

assert p_disease_given_pos >= 0
assert p_disease_given_pos <= 1

---
## Part 3: Normal Distribution [Suggested time: 20 minutes]
---
In this part, you will analyze check totals at a TexMex restaurant. We know that the population distribution of check totals for the TexMex restaurant is normally distributed with a mean of \\$20 and a standard deviation of \\$3.

### 3.1) Create a numeric variable `z_score_26` containing the z-score for a \\$26 check.

In [ ]:
###defining the variable
mean = 20
std_dev = 3
x = 26

In [ ]:
# CodeGrade step3.1
# Replace None with appropriate code


z_score_26 = (x - mean) / std_dev
print(z_score_26)

2.0


In [ ]:
# This test confirms that you have created a numeric variable named z_score_26

assert isinstance(z_score_26, Number)

### 3.2) Create a numeric variable `p_under_26` containing the approximate proportion of all checks that are less than \\$26.

Hint: Use the answer from the previous question along with the empirical rule, a Python function, or this [z-table](https://www.math.arizona.edu/~rsims/ma464/standardnormaltable.pdf).

In [ ]:
from scipy.stats import norm

z_score_26 = 2.0

In [ ]:
# CodeGrade step3.2
# Replace None with appropriate code

p_under_26 = norm.cdf(z_score_26)
print(p_under_26)

0.9772498680518208


In [ ]:
# This test confirms that you have created a numeric variable named p_under_26

assert isinstance(p_under_26, Number)

# These tests confirm that p_under_26 is a value between 0 and 1

assert p_under_26 >= 0
assert p_under_26 <= 1

### 3.3) Create numeric variables `conf_low` and `conf_high` containing the lower and upper bounds (respectively) of a 95% confidence interval for the mean of one waiter's check amounts using the information below.

One week, a waiter gets 100 checks with a mean of \\$19 and a standard deviation of \\$3.

In [ ]:
z_95 = 1.96  # Z-value for 95% confidence interval
# Standard error
SE = std / (n ** 0.5)

# Margin of error
ME = z_95 * SE

In [ ]:
# CodeGrade step3.3
# Replace None with appropriate code

n = 100
mean = 19
std = 3

conf_low = mean - ME
conf_high = mean + ME

In [ ]:
print(conf_low, conf_high)

18.412 19.588


In [ ]:
# These tests confirm that you have created numeric variables named conf_low and conf_high

assert isinstance(conf_low, Number)
assert isinstance(conf_high, Number)

# This test confirms that conf_low is below conf_high

assert conf_low < conf_high

# These statements print your answers for reference to help answer the next question

print('The lower bound of the 95% confidence interval is {}'.format(conf_low))
print('The upper bound of the 95% confidence interval is {}'.format(conf_high))

The lower bound of the 95% confidence interval is 18.412
The upper bound of the 95% confidence interval is 19.588


### 3.4) Short Answer: Interpret the 95% confidence interval you just calculated in Question 1.3.

# Your answer here
We are 95% confident that the true mean check amount for this waiter lies between $18.41 and $19.59.


---
## Part 4: Statistical Testing [Suggested time: 20 minutes]
---
The TexMex restaurant recently introduced queso to its menu.

We have a random sample containing 2000 check totals, all from different customers: 1000 check totals for orders without queso ("no queso") and 1000 check totals for orders with queso ("queso").

In the cell below, we load the sample data for you into the arrays `no_queso` and `queso` for the "no queso" and "queso" order check totals, respectively.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving queso.pkl to queso.pkl
Saving no_queso.pkl to no_queso.pkl


In [ ]:
# Run this cell without changes

# Load the sample data
no_queso = pickle.load(open('./no_queso.pkl', 'rb'))
queso = pickle.load(open('./queso.pkl', 'rb'))

### 4.1) Short Answer: State null and alternative hypotheses to use for testing whether customers who order queso spend different amounts of money from customers who do not order queso.

Null Hypothesis. Customers who order queso spend, on average, the same amount as customers who do not order queso.
Alternative Hypothesis. Customers who order queso spend, on average, a different amount than customers who do not order queso



### 4.2) Short Answer: What would it mean to make a Type I error for this specific hypothesis test?

Your answer should be _specific to this context,_  not a general statement of what Type I error is.

Concluding that customers who order queso spend a different amount of money than customers who do not, when in reality, there is no difference in their average spending.



### 4.3) Create a numeric variable `p_value` containing the p-value associated with a statistical test of your hypotheses.

You must identify and implement the correct statistical test for this scenario. You can assume the two samples have equal variances.

Hint: Use `scipy.stats` to calculate the answer - it has already been imported as `stats`. Relevant documentation can be found [here](https://docs.scipy.org/doc/scipy/reference/stats.html#statistical-tests).

In [ ]:
# CodeGrade step4.3
# Replace None with appropriate code

p_value = stats.ttest_ind(queso, no_queso, equal_var=True)

print(p_value)

TtestResult(statistic=np.float64(45.16857748646329), pvalue=np.float64(1.29670967092511e-307), df=np.float64(1998.0))


In [ ]:
# Two-sample t-test assuming equal variances
t_stat, p_value = stats.ttest_ind(queso, no_queso, equal_var=True)

# Ensure p_value is numeric
p_value = float(p_value)

print(p_value)

1.29670967092511e-307


In [ ]:
# This test confirms that you have created a numeric variable named p_value

assert isinstance(p_value, Number)

### 4.4) Short Answer: Can you reject the null hypothesis using a significance level of $\alpha$ = 0.05? Explain why or why not.

Yes,reject the null hypothesis at a significance level of α=0.05.
The null hypothesis states that the mean check totals for orders with queso and without queso are equal.
From the t-test, the p-value is extremely small (much less than 0.05).
Since p_value < 𝛼 we have strong evidence against the null hypothesis.
